In [1]:
import pandas as pd 
import numpy as np 
from scipy.stats import zscore 
import os 

In [2]:
train_path = r"D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\processed\\selected_features_train.csv"
# val_path = r"D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\processed\\selected_features_val.csv"
test_path = r"D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\processed\\selected_features_test.csv"
output_dir = r"D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\extreme_outliers"

In [3]:
os.makedirs(output_dir, exist_ok=True)

In [4]:
df_train = pd.read_csv(train_path)
# df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

In [5]:
# Function to Idnetify and manage outliers 
def handle_extreme_outliers(dataset, name): 
    target = dataset['log_H2_W% (max)'].values
    orig_target = np.expm1(target)
    z_scores = zscore(orig_target)
    extreme_outliers = dataset[abs(z_scores) > 3]
    n_outliers = len(extreme_outliers)
    print(f"{name} Extream Outliers (>3 Z-score): {n_outliers}")
    if n_outliers > 0:
        output_path = os.path.join(output_dir, f'{name}_extreme_outliers.csv')
        extreme_outliers.to_csv(output_path, index=False)
        print(f"Extreme outliers for {name} saved to {output_path}")
    
    
    df_capped = dataset.copy()
    
    cap_value = np.percentile(orig_target, 85)
    z_scores_capped = zscore(orig_target)
    df_capped.loc[abs(z_scores_capped) > 3 , 'log_H2_W% (max)'] = np.log1p(cap_value)
    
    return df_capped, extreme_outliers

In [6]:
for dataset, name in [(df_train, 'train'), (df_test, 'test')]:
    df_capped, extreme_outliers = handle_extreme_outliers(dataset, name)
    print(f"{name} shape after capping extreme outliers: {df_capped.shape}")
    
    if len(extreme_outliers) > 0:
        no_outliers_path = os.path.join(output_dir, f'{name}_no_extreme_outliers.csv')
        capped_path = os.path.join(output_dir, f'{name}_capped_extreme_outliers.csv')
        df_capped.to_csv(capped_path, index=False)
        print(f"{name} datasets saved to {no_outliers_path} and {capped_path}")

print(f"Extreme outlier analysis saved to {output_dir}")

train Extream Outliers (>3 Z-score): 16
Extreme outliers for train saved to D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\extreme_outliers\train_extreme_outliers.csv
train shape after capping extreme outliers: (893, 22)
train datasets saved to D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\extreme_outliers\train_no_extreme_outliers.csv and D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\extreme_outliers\train_capped_extreme_outliers.csv
test Extream Outliers (>3 Z-score): 11
Extreme outliers for test saved to D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\extreme_outliers\test_extreme_outliers.csv
test shape after capping extreme outliers: (386, 22)
test datasets saved to D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\extreme_outliers\test_no_extreme_outliers.csv and D:\\Hydride_Machine_learning_project\\Machine_Model\\data\\extreme_outliers\test_capped_extreme_outliers.csv
Extreme outlier analysis saved to D:\\Hydride_Machine